# 目的
* 参考文献文字列と候補文献のトークン化

## <font color="red">要実行</font>

In [1]:
# モジュールの読み込み
from pathlib import Path
import sys
import pysolr

TOOLS_DIR = Path.cwd().resolve().parent / "8プロジェクト関連" / "8-3共有プログラム"
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

import wakati
import generate_query

# 接続文字列
solr_url = "http://localhost:8983/solr/jalc"

# pysolrのクライアントの初期化
solr = pysolr.Solr(solr_url,timeout=10)

## 動作確認
* どのようにトークン化されていくのかを確認

In [2]:
from pathlib import Path
import sys

TOOLS_DIR = Path.cwd().resolve().parent / "8プロジェクト関連" / "8-3共有プログラム"
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

import wakati
import generate_query
#sample_text = r"Apache Solr の本についての考察:谷川明英:2025/06/22"
sample_text = r"Apache Solrの本について"
print("---参考文献文字列---")
print(sample_text)

print("---正規化処理---")
normalized_text = wakati.normalize_text(sample_text)
print(normalized_text)

print("---分割処理---")
splited_text = wakati.split_text(normalized_text)
print(splited_text)

print("---トークン化処理---")
token = wakati.get_token(sample_text)
print(token)

---参考文献文字列---
Apache Solrの本について
---正規化処理---
apache solr|の本について
---分割処理---
['apache solr', 'の本について']
---トークン化処理---
['apache', 'solr', 'apache solr', 'の本', '本に', 'につ', 'つい', 'いて']


In [ ]:
title = r""
print(wakati.normalize_text(title))
print(wakati.split_text(wakati.normalize_text(title)))

1|日|5|分で親子関係が変わる!育児が楽になる! pcit|から学ぶ子育て | 加茂登志子著
['5', '分で親子関係が変わる', '育児が楽になる', 'pcit', 'から学ぶ子育て', '加茂登志子著']


In [31]:
english_text = "1 2 3"
print(wakati.slice_word_ngram(english_text,1))

['1', '2', '3']


## トークン生成及びスコア算出
* RCとCCでトークン集合が変化する

### RC

In [5]:
# リスト型データの各要素をトークン化して結合
def list_to_token(lis):
    tokens = []
    for val in lis:
        tokens += wakati.get_token(val)
    return tokens

# sim_r 算出用の候補文献のトークン集合を得る
def get_sim_r_token(document):
    tokens = []
    # トークン化するフィールド一覧
    field_names = ["creator","title","journal","issued","volume_issue","page_range"]
    # 各フィールドをトークン化して結合
    for field_name in field_names:
        tokens += list(set(list_to_token(document[field_name])))    
    return tokens    

# sim_r の算出
def calc_sim_r(reference_token,sim_r_token):
    # sim_r スコア
    sim_r_score = 0
    # 両方のトークン集合の共通要素の数
    common_token_num = 0

    for token in reference_token:
        if str(token) in sim_r_token:
            common_token_num += 1
        else:
            print(f"含まれなかったトークン：{token}")    

    sim_r_score = common_token_num / len(reference_token)

    return sim_r_score            

# 以下本処理

# 参考文献文字列
reference = input()
escaped_query = generate_query.escape_solr_query(reference)

# 検索オプションの設定
search_options = {
    'q.op':'OR',
    'fl':'doi,first_author,creator,title,journal,issued,volume_issue,page_range,score',
    'df':'jalcdata',
    'rows':2
}

# 検索
results = solr.search(escaped_query,**search_options)

# 参考文献文字列のトークン化
reference_token = list(set(wakati.get_token(reference)))
print("---参考文献文字列---")
print(reference)
print("---参考文献文字列のトークン集合---")
print(reference_token)
print(f"|tokens_ref| = {len(reference_token)}")

# 候補文献のトークン化(sim_r)
print("---候補文献のトークン化(sim_r)---")
for result in results:
    sim_r_token = get_sim_r_token(result)
    print(sim_r_token)
    print(f"|RC_tokens| = {len(sim_r_token)}")
    print("---sim_rスコア---")
    sim_r_score = calc_sim_r(reference_token,sim_r_token)
    print(sim_r_score)
    print(result)    


---参考文献文字列---
平井克之，林貴宏：研究資金の公募情報を推薦するシステムの構築―プレアワード業務の効率化へ，情報の科学と技術，Vol. 68, No.6, pp. 298-303 (2018).
---参考文献文字列のトークン集合---
['アワ', '68', '資金', '金の', 'ワー', '効率', '2018', 'るシ', '構築', '科学', 'ステ', '化へ', '学と', '林貴', '率化', 'ード', '研究', '情報', 'シス', '究資', 'ド業', 'ムの', '技術', 'する', 'テム', '303', '公募', 'レア', '報の', 'を推', 'と技', '薦す', 'の効', '298', '克之', '井克', 'の構', 'プレ', '平井', 'の公', '6', '募情', 'の科', '報を', '業務', '務の', '貴宏', '推薦']
|tokens_ref| = 48
---候補文献のトークン化(sim_r)---
['hirai', 'k', '林', 'katsuyuki', 'hayashi', '克之', '井克', '貴宏', 't', '平井', '林貴', 'takahiro', 'アワ', '資金', '金の', 'ワー', '効率', 'award efficiency', 'grants', 'るシ', '構築', 'ステ', 'research', '化へ', 'for', '率化', 'ード', '研究', '情報', 'recommendation system', 'シス', '究資', 'ド業', 'ムの', 'system', 'possibility', 'pre', 'する', 'テム', '公募', 'research grants', 'レア', 'を推', '薦す', 'の効', 'recommendation', 'プレ', 'efficiency', 'の構', 'improving', 'of', 'の公', '募情', 'award', '報を', '業務', '務の', 'a', '推薦', 'information', 'science', 'technology', '学と', 'joho', '情報', 'associat

### CC

In [29]:
# リスト型データの各要素のトークン化
def list_to_token(lis):
    tokens = []
    for val in lis:
        tokens.append(wakati.get_token(val))
    return tokens

# sim_p 算出用のトークン集合を得る
def get_sim_p_token(document):
    sim_p_token = []
    # トークン化するフィールド名一覧
    field_names = ["first_author","title","journal","issued","volume_issue","page_range"]
    # 各フィールドをトークン化
    for field in field_names:
        sim_p_token.append(list_to_token(document[field]))
    return sim_p_token

# sim_p 算出関数
def calc_sim_p(reference_token,sim_p_token):
    sim_p_score = 0
    for i in range(len(sim_p_token)):
        max_score = 0
        for j in range(len(sim_p_token[i])):
            common_token_num = 0
            for token in sim_p_token[i][j]:
                if token in reference_token:
                    common_token_num += 1
            tmp = common_token_num / len(sim_p_token[i][j])
            if tmp > max_score:
                max_score = tmp
        print(max_score)
        sim_p_score += max_score
    return sim_p_score / 6                   
    
# 以下本処理
    
# 参考文献文字列
reference = "平井克之，林貴宏：研究資金の公募情報を推薦するシステムの構築―プレアワード業務の効率化へ，情報の科学と技術，Vol. 68, No.6, pp. 298-303 (2018)."
print("---参考文献文字列---")
print(reference)

# エスケープ処理
escaped_query = generate_query.escape_solr_query(reference)

# 検索オプションの設定
search_options = {
    'q.op':'OR',
    'fl':'doi,first_author,creator,title,journal,issued,volume_issue,page_range,score',
    'df':'jalcdata',
    'rows':2
}

# 検索
results = solr.search(escaped_query,**search_options)

# 参考文献文字列のトークン化
reference_token = list(set(wakati.get_token(reference)))
print("---参考文献文字列のトークン化---")
print(reference_token)

# 候補文献のトークン化及びsim_p算出
for result in results:
    print("---候補文献のトークン化---")
    sim_p_token = get_sim_p_token(result)
    print(sim_p_token)
    print("---sim_pの算出---")
    sim_p_score = calc_sim_p(reference_token,sim_p_token)
    print(sim_p_score)


---参考文献文字列---
平井克之，林貴宏：研究資金の公募情報を推薦するシステムの構築―プレアワード業務の効率化へ，情報の科学と技術，Vol. 68, No.6, pp. 298-303 (2018).
---参考文献文字列のトークン化---
['アワ', '68', '資金', '金の', 'ワー', '効率', '2018', 'るシ', '構築', '科学', 'ステ', '化へ', '学と', '林貴', '率化', 'ード', '研究', '情報', 'シス', '究資', 'ド業', 'ムの', '技術', 'する', 'テム', '303', '公募', 'レア', '報の', 'を推', 'と技', '薦す', 'の効', '298', '克之', '井克', 'の構', 'プレ', '平井', 'の公', '6', '募情', 'の科', '報を', '業務', '務の', '貴宏', '推薦']
---候補文献のトークン化---
[[['平井', '井克', '克之'], ['hirai'], ['平井']], [['研究', '究資', '資金', '金の', 'の公', '公募', '募情', '情報', '報を', 'を推', '推薦', '薦す', 'する', 'るシ', 'シス', 'ステ', 'テム', 'ムの', 'の構', '構築', 'プレ', 'レア', 'アワ', 'ワー', 'ード', 'ド業', '業務', '務の', 'の効', '効率', '率化', '化へ'], ['a', 'recommendation', 'system', 'recommendation system', 'of', 'research', 'grants', 'research grants', 'possibility', 'for', 'improving', 'pre', 'award', 'efficiency', 'award efficiency']], [['joho', 'kagaku', 'to', 'gijutsu'], ['情報', '報の', 'の科', '科学', '学と', 'と技', '技術'], ['the', 'journal', 'of', 'information', 'science', 'info

In [28]:
text = input()
text = text.replace("'","")
text = text.replace(", ","，")
text = text.replace("[","{")
text = text.replace("]","}")
print(text)

{{298，302}}
